# Claim Grounding Pilot

Notebook-first pilot for WU-E1. This notebook fixes the pilot scope to the first five `data/dev_subset.csv` records, uses the saved PDF-file run artifact at `artifacts/runs/20260306_124634_dev_subset_pdf_file.json`, and restricts review to `geospatial_info_dataset`, `data_type`, and `species`.

The grounding columns produced here are intentionally lightweight lexical candidates, not the production evidence pass. Their purpose is to make the first five records reviewable now, surface a few representative examples, and give WU-E2 a concrete table shape to target.


In [1]:
from __future__ import annotations

import ast
import html
import json
import math
import re
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root from the current notebook path.")


ROOT = find_repo_root(Path.cwd())
TARGET_FIELDS = ["geospatial_info_dataset", "data_type", "species"]
RUN_ARTIFACT_PATH = ROOT / "artifacts" / "runs" / "20260306_124634_dev_subset_pdf_file.json"
GT_PATH = ROOT / "data" / "dataset_092624_validated.xlsx"
DEV_SUBSET_PATH = ROOT / "data" / "dev_subset.csv"
OUTPUT_PATH = ROOT / "artifacts" / "claim_grounding_pilot_first5.csv"

GROUNDING_CUES = {
    "data_type": {
        "abundance": ["abundance", "density", "count", "counts", "estimated"],
        "density": ["density", "densities", "isodar"],
        "presence-only": ["presence", "occurrence", "identified", "observed"],
        "presence-absence": ["presence absence", "presence-absence"],
        "genetic_analysis": ["genetic", "microsatellite", "genotyp", "loci"],
        "distribution": ["distribution", "range expansion", "expanding"],
        "traits": ["movement", "fitness", "reproductive success", "behavio"],
        "ecosystem_structure": ["habitat", "landscape composition", "forest edge"],
        "time_series": ["over two years", "successive years", "consecutive years", "2004 2011", "1999 2011"],
    },
    "geospatial_info_dataset": {
        "sample": ["x meter", "y meter", "coordinates", "live trap locations", "sample id"],
        "site": ["river", "region", "regions", "park", "study area", "sites"],
        "maps": ["map", "maps"],
        "administrative_units": ["quebec", "canada", "ontario", "charlevoix", "estrie", "monteregie", "shawinigan"],
        "range": ["range", "home range", "distributional limit"],
        "site_ids": ["sample id", "id"],
    },
}


def clean_text(text: object) -> str:
    if text is None:
        return ""
    if isinstance(text, float) and math.isnan(text):
        return ""
    cleaned = html.unescape(str(text))
    cleaned = re.sub(r"<[^>]+>", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned.strip()


def normalize_text(text: object) -> str:
    cleaned = clean_text(text)
    cleaned = unicodedata.normalize("NFKD", cleaned).encode("ascii", "ignore").decode("ascii")
    cleaned = cleaned.casefold()
    cleaned = re.sub(r"[^a-z0-9]+", " ", cleaned)
    return cleaned.strip()


def split_sentences(text: object) -> list[str]:
    cleaned = clean_text(text)
    parts = re.split(r"(?<=[.!?])\s+", cleaned)
    return [part.strip() for part in parts if part.strip()]


def to_atomic_list(value: object) -> list[str]:
    if value is None:
        return []
    if isinstance(value, float) and math.isnan(value):
        return []
    parsed = value
    if isinstance(value, str):
        stripped = value.strip()
        if not stripped:
            return []
        if stripped.startswith("[") and stripped.endswith("]"):
            try:
                parsed = ast.literal_eval(stripped)
            except (SyntaxError, ValueError):
                parsed = stripped
        else:
            parsed = stripped
    if isinstance(parsed, list):
        return [str(item).strip() for item in parsed if str(item).strip()]
    return [str(parsed).strip()]


def empty_grounding(reason: str) -> dict[str, str | None]:
    return {"support_type": None, "quote": None, "rationale": reason}


def draft_grounding(text: str, field_name: str, target_value: str | None) -> dict[str, str | None]:
    if not target_value:
        return empty_grounding("No claim value for this side of the comparison.")

    sentences = split_sentences(text)
    normalized_target = normalize_text(target_value)

    if field_name == "species":
        for sentence in sentences:
            if normalized_target and normalized_target in normalize_text(sentence):
                return {
                    "support_type": "explicit",
                    "quote": sentence,
                    "rationale": "Exact species string appears in the source text.",
                }

        tokens = [token for token in normalized_target.split() if len(token) > 2]
        for sentence in sentences:
            sentence_norm = normalize_text(sentence)
            hits = sum(token in sentence_norm for token in tokens)
            if hits >= min(2, len(tokens)) and hits > 0:
                return {
                    "support_type": "paraphrase",
                    "quote": sentence,
                    "rationale": "Partial lexical overlap suggests a related species mention worth manual review.",
                }

        return {
            "support_type": "unsupported",
            "quote": None,
            "rationale": "No direct species mention found by the notebook pilot heuristic.",
        }

    cues = GROUNDING_CUES.get(field_name, {}).get(target_value, [])
    for sentence in sentences:
        sentence_norm = normalize_text(sentence)
        cue_hits = [cue for cue in cues if normalize_text(cue) in sentence_norm]
        if cue_hits:
            return {
                "support_type": "paraphrase",
                "quote": sentence,
                "rationale": f"Pilot cue match for {target_value}: {', '.join(cue_hits[:2])}.",
            }

    return {
        "support_type": "unsupported",
        "quote": None,
        "rationale": "No lexical cue found; this row is a good candidate for the WU-E2 LLM grounding pass.",
    }


def bucket_row(row: pd.Series) -> str:
    if bool(row["match"]):
        return "match"
    if pd.notna(row["gt_value"]) and pd.isna(row["pred_value"]):
        return "gt_only"
    if pd.isna(row["gt_value"]) and pd.notna(row["pred_value"]):
        return "pred_only"
    return "overlap_mismatch"


def build_comparison_table(gt_frame: pd.DataFrame, run_artifact: dict, pilot_ids: list[int]) -> pd.DataFrame:
    gt_index = gt_frame.set_index("id")
    pred_index = {record["gt_record_id"]: record for record in run_artifact["records"] if record["gt_record_id"] in pilot_ids}
    rows: list[dict[str, object]] = []

    for record_id in pilot_ids:
        gt_row = gt_index.loc[record_id]
        pred_row = pred_index[record_id]
        source_text = pred_row.get("input_text") or pred_row.get("abstract") or ""

        for field_name in TARGET_FIELDS:
            gt_values = to_atomic_list(gt_row.get(field_name))
            pred_values = to_atomic_list((pred_row.get("output") or {}).get(field_name))
            gt_map = {normalize_text(value): value for value in gt_values}
            pred_map = {normalize_text(value): value for value in pred_values}
            atomic_keys = sorted(key for key in set(gt_map) | set(pred_map) if key)

            if not atomic_keys:
                rows.append({
                    "gt_record_id": record_id,
                    "doi": pred_row.get("record_id"),
                    "title": pred_row.get("title") or gt_row.get("title"),
                    "field": field_name,
                    "target_value": None,
                    "match": None,
                    "gt_value": None,
                    "pred_value": None,
                    "gt_support_type": None,
                    "gt_quote": None,
                    "gt_rationale": "No GT or prediction value for this field on this record.",
                    "pred_support_type": None,
                    "pred_quote": None,
                    "pred_rationale": "No GT or prediction value for this field on this record.",
                    "source_text": clean_text(source_text),
                })
                continue

            for atomic_key in atomic_keys:
                gt_value = gt_map.get(atomic_key)
                pred_value = pred_map.get(atomic_key)
                gt_grounding = draft_grounding(source_text, field_name, gt_value)
                pred_grounding = draft_grounding(source_text, field_name, pred_value)
                rows.append({
                    "gt_record_id": record_id,
                    "doi": pred_row.get("record_id"),
                    "title": pred_row.get("title") or gt_row.get("title"),
                    "field": field_name,
                    "target_value": gt_value or pred_value,
                    "match": atomic_key in gt_map and atomic_key in pred_map,
                    "gt_value": gt_value,
                    "pred_value": pred_value,
                    "gt_support_type": gt_grounding["support_type"],
                    "gt_quote": gt_grounding["quote"],
                    "gt_rationale": gt_grounding["rationale"],
                    "pred_support_type": pred_grounding["support_type"],
                    "pred_quote": pred_grounding["quote"],
                    "pred_rationale": pred_grounding["rationale"],
                    "source_text": clean_text(source_text),
                })

    comparison = pd.DataFrame(rows)
    comparison["bucket"] = comparison.apply(bucket_row, axis=1)
    comparison["record_field"] = comparison["gt_record_id"].astype(str) + " · " + comparison["field"]
    return comparison.sort_values(["gt_record_id", "field", "bucket", "target_value"], ascending=[True, True, True, True]).reset_index(drop=True)


def compact_view(frame: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "gt_record_id",
        "field",
        "bucket",
        "target_value",
        "gt_support_type",
        "pred_support_type",
        "gt_quote",
        "pred_quote",
    ]
    return frame[cols].reset_index(drop=True)


def show_record_example(frame: pd.DataFrame, record_id: int, field_name: str | None = None, max_rows: int = 12) -> None:
    subset = frame[frame["gt_record_id"] == record_id].copy()
    if field_name is not None:
        subset = subset[subset["field"] == field_name].copy()
    if subset.empty:
        display(Markdown(f"No rows found for record `{record_id}`."))
        return

    row0 = subset.iloc[0]
    heading = f"### Record {record_id} · {row0['field']}" if field_name else f"### Record {record_id}"
    display(Markdown(heading))
    display(Markdown(f"**DOI:** `{row0['doi']}`  \\n**Title:** {row0['title']}"))
    display(Markdown("**Source Text Excerpt**"))
    display(Markdown(row0["source_text"][:1800] + ("..." if len(row0["source_text"]) > 1800 else "")))
    display(compact_view(subset.head(max_rows)))


pd.set_option("display.max_colwidth", 160)


In [2]:
dev_subset = pd.read_csv(DEV_SUBSET_PATH)
pilot_ids = dev_subset.head(5)["id"].astype(int).tolist()

gt_frame = pd.read_excel(GT_PATH)
gt_frame = gt_frame[gt_frame["id"].isin(pilot_ids)].copy()

run_artifact = json.loads(RUN_ARTIFACT_PATH.read_text(encoding="utf-8"))
pilot_records = pd.DataFrame(
    [
        {
            "gt_record_id": record["gt_record_id"],
            "doi": record["record_id"],
            "title": record.get("title"),
            "notes": dev_subset.loc[dev_subset["id"] == record["gt_record_id"], "notes"].iloc[0],
        }
        for record in run_artifact["records"]
        if record["gt_record_id"] in pilot_ids
    ]
).sort_values("gt_record_id")

pilot_records


,gt_record_id,doi,title,notes
0,9,10.1371/journal.pone.0128238,Data from: Resampling method for applying density-dependent habitat selection theory to wildlife surveys,vernacular_species; multi_species_list; gt_ambiguity(1_null_bools)
1,19,10.1093/jhered/esw073,Data from: The genetic signature of range expansion in a disease vector - the black-legged tick,time_series=True; has_species
2,27,10.1111/ddi.12496,Data from: Patchy distribution and low effective population size raise concern for an at-risk top predator,threatened_species=True; vernacular_species; multi_species_list; gt_ambiguity(1_null_bools)
3,29,10.1002/ece3.1476,"Data from: Ecological gradients driving the distribution of four Ericaceae in boreal Quebec, Canada",bias_north_south=True; multi_species_list; gt_ambiguity(1_null_bools)
4,30,10.1186/s40462-016-0079-4,"Data from: Caribou, water, and ice – fine-scale movements of a migratory arctic ungulate in the context of climate change",threatened_species=True; bias_north_south=True; has_species; gt_ambiguity(1_null_bools)


In [3]:
comparison_df = build_comparison_table(gt_frame=gt_frame, run_artifact=run_artifact, pilot_ids=pilot_ids)
comparison_df.to_csv(OUTPUT_PATH, index=False)

compact_view(comparison_df.head(15))


,gt_record_id,field,bucket,target_value,gt_support_type,pred_support_type,gt_quote,pred_quote
0,9,data_type,gt_only,density,paraphrase,None,Data from: Resampling method for applying density-dependent habitat selection theory to wildlife surveys Isodar theory can be used to evaluate fitness conse...,None
1,9,data_type,pred_only,abundance,None,paraphrase,None,Data from: Resampling method for applying density-dependent habitat selection theory to wildlife surveys Isodar theory can be used to evaluate fitness conse...
2,9,geospatial_info_dataset,match,sample,paraphrase,paraphrase,"The columns ""X_METER"" and ""Y_METER"" are X and Y projected coordinates (in meters) of live-trap locations.","The columns ""X_METER"" and ""Y_METER"" are X and Y projected coordinates (in meters) of live-trap locations."
3,9,geospatial_info_dataset,pred_only,administrative_units,None,paraphrase,None,Capture data of raccoons and striped skunks in Montérégie and Estrie regions Capture data of raccoons and striped skunks during wildlife surveys in Montérég...
4,9,geospatial_info_dataset,pred_only,maps,None,unsupported,None,None
5,9,geospatial_info_dataset,pred_only,site,None,paraphrase,None,Capture data of raccoons and striped skunks in Montérégie and Estrie regions Capture data of raccoons and striped skunks during wildlife surveys in Montérég...
6,9,species,gt_only,raccoons,explicit,None,"We applied this method to abundance data of raccoons and striped skunks, two of the main hosts of rabies virus in North America.",None
7,9,species,gt_only,striped skunks,explicit,None,"We applied this method to abundance data of raccoons and striped skunks, two of the main hosts of rabies virus in North America.",None
8,9,species,pred_only,raccoon (Procyon lotor): 610 individuals captured,None,paraphrase,None,Capture_data_Tardy_et_al.xlsx Wildlife survey; Procyon lotor; Mephitis mephitis; Striped skunk; Raccoon; capture Canada; Montérégie and Estrie; Québec
9,9,species,pred_only,striped skunk (Mephitis mephitis): 121 individuals captured,None,paraphrase,None,"We applied this method to abundance data of raccoons and striped skunks, two of the main hosts of rabies virus in North America."


## Quick Navigation

Start with the summaries below, then jump into the example slices. The buckets are:

- `match`: exact atomic overlap between GT and prediction
- `gt_only`: GT claim missing from the prediction side
- `pred_only`: prediction claim not present in GT
- `overlap_mismatch`: reserved for future relaxed matching; should be rare in this exact-match pilot


In [4]:
comparison_df.groupby(["field", "bucket"]).size().unstack(fill_value=0)


bucket,gt_only,match,pred_only
field,,,
data_type,3,4,7
geospatial_info_dataset,0,4,19
species,5,5,12


In [5]:
comparison_df.groupby(["field", "pred_support_type"], dropna=False).size().unstack(fill_value=0)


pred_support_type,explicit,paraphrase,unsupported,NaN
field,,,,
data_type,0,7,4,3
geospatial_info_dataset,0,16,7,0
species,10,6,1,5


## Example Views

These are the three patterns still worth reading first in the PDF-file pilot:

- record `9`: `data_type` disagreement plus species values bloated with counts
- record `27`: species over-extraction around canid labels remains clear in the PDF-file output
- record `30`: strong `data_type` drift and geospatial over-prediction remain easy to inspect


In [6]:
show_record_example(comparison_df, 9, "data_type")
show_record_example(comparison_df, 9, "species")


### Record 9 · data_type

**DOI:** `10.1371/journal.pone.0128238`  \n**Title:** Data from: Resampling method for applying density-dependent habitat selection theory to wildlife surveys

**Source Text Excerpt**

Data from: Resampling method for applying density-dependent habitat selection theory to wildlife surveys Isodar theory can be used to evaluate fitness consequences of density-dependent habitat selection by animals. A typical habitat isodar is a regression curve plotting competitor densities in two adjacent habitats when individual fitness is equal. Despite the increasing use of habitat isodars, their application remains largely limited to areas composed of pairs of adjacent habitats that are defined a priori. We developed a resampling method that uses data from wildlife surveys to build isodars in heterogeneous landscapes without having to predefine habitat types. The method consists in randomly placing blocks over the survey area and dividing those blocks in two adjacent sub-blocks of the same size. Animal abundance is then estimated within the two sub-blocks. This process is done 100 times. Different functional forms of isodars can be investigated by relating animal abundance and differences in habitat features between sub-blocks. We applied this method to abundance data of raccoons and striped skunks, two of the main hosts of rabies virus in North America. Habitat selection by raccoons and striped skunks depended on both conspecific abundance and the difference in landscape composition and structure between sub-blocks. When conspecific abundance was low, raccoons and striped skunks favored areas with relatively high proportions of forests and anthropogenic features, respectively. Under high conspecific abundance, however, both species preferred areas with rather large corn-forest edge densities and corn field proportions. Based on random sampling techniques, we provide a robust method that is applicable to a broad range of species, including medium- to large-sized ma...

,gt_record_id,field,bucket,target_value,gt_support_type,pred_support_type,gt_quote,pred_quote
0,9,data_type,gt_only,density,paraphrase,None,Data from: Resampling method for applying density-dependent habitat selection theory to wildlife surveys Isodar theory can be used to evaluate fitness conse...,None
1,9,data_type,pred_only,abundance,None,paraphrase,None,Data from: Resampling method for applying density-dependent habitat selection theory to wildlife surveys Isodar theory can be used to evaluate fitness conse...


### Record 9 · species

**DOI:** `10.1371/journal.pone.0128238`  \n**Title:** Data from: Resampling method for applying density-dependent habitat selection theory to wildlife surveys

**Source Text Excerpt**

Data from: Resampling method for applying density-dependent habitat selection theory to wildlife surveys Isodar theory can be used to evaluate fitness consequences of density-dependent habitat selection by animals. A typical habitat isodar is a regression curve plotting competitor densities in two adjacent habitats when individual fitness is equal. Despite the increasing use of habitat isodars, their application remains largely limited to areas composed of pairs of adjacent habitats that are defined a priori. We developed a resampling method that uses data from wildlife surveys to build isodars in heterogeneous landscapes without having to predefine habitat types. The method consists in randomly placing blocks over the survey area and dividing those blocks in two adjacent sub-blocks of the same size. Animal abundance is then estimated within the two sub-blocks. This process is done 100 times. Different functional forms of isodars can be investigated by relating animal abundance and differences in habitat features between sub-blocks. We applied this method to abundance data of raccoons and striped skunks, two of the main hosts of rabies virus in North America. Habitat selection by raccoons and striped skunks depended on both conspecific abundance and the difference in landscape composition and structure between sub-blocks. When conspecific abundance was low, raccoons and striped skunks favored areas with relatively high proportions of forests and anthropogenic features, respectively. Under high conspecific abundance, however, both species preferred areas with rather large corn-forest edge densities and corn field proportions. Based on random sampling techniques, we provide a robust method that is applicable to a broad range of species, including medium- to large-sized ma...

,gt_record_id,field,bucket,target_value,gt_support_type,pred_support_type,gt_quote,pred_quote
0,9,species,gt_only,raccoons,explicit,None,"We applied this method to abundance data of raccoons and striped skunks, two of the main hosts of rabies virus in North America.",None
1,9,species,gt_only,striped skunks,explicit,None,"We applied this method to abundance data of raccoons and striped skunks, two of the main hosts of rabies virus in North America.",None
2,9,species,pred_only,raccoon (Procyon lotor): 610 individuals captured,None,paraphrase,None,Capture_data_Tardy_et_al.xlsx Wildlife survey; Procyon lotor; Mephitis mephitis; Striped skunk; Raccoon; capture Canada; Montérégie and Estrie; Québec
3,9,species,pred_only,striped skunk (Mephitis mephitis): 121 individuals captured,None,paraphrase,None,"We applied this method to abundance data of raccoons and striped skunks, two of the main hosts of rabies virus in North America."


In [7]:
show_record_example(comparison_df, 27, "species")


### Record 27 · species

**DOI:** `10.1111/ddi.12496`  \n**Title:** Data from: Patchy distribution and low effective population size raise concern for an at-risk top predator

**Source Text Excerpt**

Data from: Patchy distribution and low effective population size raise concern for an at-risk top predator Aim: Understanding carnivore distribution is important for management decisions that aim to restore naturally-regulated ecosystems and preserve biodiversity. Eastern Wolves, a species at risk in Canada, are centralized in Algonquin Provincial Park and their ability to disperse and establish themselves elsewhere is limited by human-caused mortality associated with hunting, trapping, and vehicle collisions. Here, we refine our understanding of Eastern Wolf distribution and provide the first estimates of their effective population size. Location: Southern Ontario and Gatineau Quebec. Methods: We used noninvasive samples, as well as blood samples archived from other research projects, collected between 2010 – 2014 to generate autosomal microsatellite genotypes at 12 loci for 98 Canis individuals. We utilized Bayesian and multivariate clustering analyses to identify Eastern Wolves in regions that were previously unsampled. Both linkage disequilibrium and temporal approaches were used to estimate effective population size of Eastern Wolves. Results: Assignment tests identified 34 individuals as Eastern Wolves, primarily in or near two provincial parks: Killarney and Queen Elizabeth II Wildlands. Eastern Coyotes were identified in Bon Echo Provincial Park, Frontenac Provincial Park, and Gatineau Park, whereas many of the samples were admixed among the different Canis types. Effective population size (Ne) estimates ranged from 24.3 – 122.1 with a harmonic mean of 45.6. Main Conclusions: The identification of Eastern Wolves in the regions of Killarney and Queen Elizabeth II Wildlands Provincial Parks extends the range of Eastern Wolves north of the French River and southwar...

,gt_record_id,field,bucket,target_value,gt_support_type,pred_support_type,gt_quote,pred_quote
0,27,species,gt_only,Eastern coyote,explicit,None,"Eastern Coyotes were identified in Bon Echo Provincial Park, Frontenac Provincial Park, and Gatineau Park, whereas many of the samples were admixed among th...",None
1,27,species,gt_only,eastern wolf,explicit,None,"Here, we refine our understanding of Eastern Wolf distribution and provide the first estimates of their effective population size.",None
2,27,species,pred_only,Domestic Dog (Canis familiaris),None,paraphrase,None,Canis lupus familiaris; Canis lycaon; eastern wolf; predator distribution; Canis lycaon sp.
3,27,species,pred_only,Eastern Coyote (Canis latrans),None,paraphrase,None,"Eastern Coyotes were identified in Bon Echo Provincial Park, Frontenac Provincial Park, and Gatineau Park, whereas many of the samples were admixed among th..."
4,27,species,pred_only,Eastern Wolf (Canis lycaon),None,paraphrase,None,"Here, we refine our understanding of Eastern Wolf distribution and provide the first estimates of their effective population size."
5,27,species,pred_only,Great Lakes-Boreal Wolf (Canis lupus × lycaon),None,paraphrase,None,Canis lupus familiaris; Canis lycaon; eastern wolf; predator distribution; Canis lycaon sp.


In [8]:
show_record_example(comparison_df, 30, "data_type")
show_record_example(comparison_df, 30, "geospatial_info_dataset")


### Record 30 · data_type

**DOI:** `10.1186/s40462-016-0079-4`  \n**Title:** Data from: Caribou, water, and ice – fine-scale movements of a migratory arctic ungulate in the context of climate change

**Source Text Excerpt**

Data from: Caribou, water, and ice – fine-scale movements of a migratory arctic ungulate in the context of climate change Freshwater lakes and rivers of the Northern Hemisphere have been freezing increasingly later and thawing increasingly earlier during the last century. With reduced temporal periods during which ice conditions are favourable for locomotion, freshwater bodies could become impediments to the inter-patch movements, dispersion, or migration of terrestrial animals that use ice-covered lakes and rivers to move across their range. Studying the fine-scale responses of individuals to broad-scale changes in ice availability and phenology would help to understand how animals react to ongoing climate change, and contribute to the conservation and management of endangered species living in northern environments. Between 2007 and 2014, we equipped 96 migratory caribou Rangifer tarandus caribou from the Rivière-aux-Feuilles herd in northern Québec (Canada) with GPS telemetry collars and studied their space use. We measured contemporary (digital MODIS maps updated every 8 days, 2000–2014) and historical (annual observations, 1947–1985) variations in freshwater-ice availability and evaluated the concurrent responses of caribou to these changes. Ice-crossing locations The GPS locations (n = 653) of migratory caribou from the Rivière-aux-Feuilles herd that crossed lakes on ice from 2007 to 2014 in northern Quebec. IceCross_Data.xlsx Water-crossing locations The GPS locations (n = 139) of migratory caribou from the Rivière-aux-Feuilles herd that crossed lakes through water from 2007 to 2014 in northern Quebec. WaterCross_Data.xlsx Detour locations The GPS locations (n = 1119) of migratory caribou from the Rivière-aux-Feuilles herd that circumvented lakes from 2007 to 201...

,gt_record_id,field,bucket,target_value,gt_support_type,pred_support_type,gt_quote,pred_quote
0,30,data_type,gt_only,presence-only,unsupported,None,None,None
1,30,data_type,pred_only,distribution,None,unsupported,None,None
2,30,data_type,pred_only,other,None,unsupported,None,None
3,30,data_type,pred_only,time_series,None,unsupported,None,None
4,30,data_type,pred_only,traits,None,paraphrase,None,"Data from: Caribou, water, and ice – fine-scale movements of a migratory arctic ungulate in the context of climate change Freshwater lakes and rivers of the..."


### Record 30 · geospatial_info_dataset

**DOI:** `10.1186/s40462-016-0079-4`  \n**Title:** Data from: Caribou, water, and ice – fine-scale movements of a migratory arctic ungulate in the context of climate change

**Source Text Excerpt**

Data from: Caribou, water, and ice – fine-scale movements of a migratory arctic ungulate in the context of climate change Freshwater lakes and rivers of the Northern Hemisphere have been freezing increasingly later and thawing increasingly earlier during the last century. With reduced temporal periods during which ice conditions are favourable for locomotion, freshwater bodies could become impediments to the inter-patch movements, dispersion, or migration of terrestrial animals that use ice-covered lakes and rivers to move across their range. Studying the fine-scale responses of individuals to broad-scale changes in ice availability and phenology would help to understand how animals react to ongoing climate change, and contribute to the conservation and management of endangered species living in northern environments. Between 2007 and 2014, we equipped 96 migratory caribou Rangifer tarandus caribou from the Rivière-aux-Feuilles herd in northern Québec (Canada) with GPS telemetry collars and studied their space use. We measured contemporary (digital MODIS maps updated every 8 days, 2000–2014) and historical (annual observations, 1947–1985) variations in freshwater-ice availability and evaluated the concurrent responses of caribou to these changes. Ice-crossing locations The GPS locations (n = 653) of migratory caribou from the Rivière-aux-Feuilles herd that crossed lakes on ice from 2007 to 2014 in northern Quebec. IceCross_Data.xlsx Water-crossing locations The GPS locations (n = 139) of migratory caribou from the Rivière-aux-Feuilles herd that crossed lakes through water from 2007 to 2014 in northern Quebec. WaterCross_Data.xlsx Detour locations The GPS locations (n = 1119) of migratory caribou from the Rivière-aux-Feuilles herd that circumvented lakes from 2007 to 201...

,gt_record_id,field,bucket,target_value,gt_support_type,pred_support_type,gt_quote,pred_quote
0,30,geospatial_info_dataset,match,sample,unsupported,unsupported,None,None
1,30,geospatial_info_dataset,pred_only,maps,None,paraphrase,None,"We measured contemporary (digital MODIS maps updated every 8 days, 2000–2014) and historical (annual observations, 1947–1985) variations in freshwater-ice a..."
2,30,geospatial_info_dataset,pred_only,range,None,paraphrase,None,"With reduced temporal periods during which ice conditions are favourable for locomotion, freshwater bodies could become impediments to the inter-patch movem..."
3,30,geospatial_info_dataset,pred_only,site,None,paraphrase,None,"Data from: Caribou, water, and ice – fine-scale movements of a migratory arctic ungulate in the context of climate change Freshwater lakes and rivers of the..."


## Follow-up Filters

Use these small slices when reviewing the pilot output:


In [9]:
compact_view(comparison_df[comparison_df["bucket"] == "pred_only"].sort_values(["field", "gt_record_id", "target_value"]))


,gt_record_id,field,bucket,target_value,gt_support_type,pred_support_type,gt_quote,pred_quote
0,9,data_type,pred_only,abundance,None,paraphrase,None,Data from: Resampling method for applying density-dependent habitat selection theory to wildlife surveys Isodar theory can be used to evaluate fitness conse...
1,19,data_type,pred_only,time_series,None,unsupported,None,None
2,29,data_type,pred_only,distribution,None,paraphrase,None,"Data from: Ecological gradients driving the distribution of four Ericaceae in boreal Quebec, Canada Understory species play a significant role in forest eco..."
3,30,data_type,pred_only,distribution,None,unsupported,None,None
4,30,data_type,pred_only,other,None,unsupported,None,None
5,30,data_type,pred_only,time_series,None,unsupported,None,None
6,30,data_type,pred_only,traits,None,paraphrase,None,"Data from: Caribou, water, and ice – fine-scale movements of a migratory arctic ungulate in the context of climate change Freshwater lakes and rivers of the..."
7,9,geospatial_info_dataset,pred_only,administrative_units,None,paraphrase,None,Capture data of raccoons and striped skunks in Montérégie and Estrie regions Capture data of raccoons and striped skunks during wildlife surveys in Montérég...
8,9,geospatial_info_dataset,pred_only,maps,None,unsupported,None,None
9,9,geospatial_info_dataset,pred_only,site,None,paraphrase,None,Capture data of raccoons and striped skunks in Montérégie and Estrie regions Capture data of raccoons and striped skunks during wildlife surveys in Montérég...


In [10]:
compact_view(comparison_df[comparison_df["pred_support_type"] == "unsupported"].sort_values(["field", "gt_record_id", "target_value"]))


,gt_record_id,field,bucket,target_value,gt_support_type,pred_support_type,gt_quote,pred_quote
0,19,data_type,pred_only,time_series,None,unsupported,None,None
1,30,data_type,pred_only,distribution,None,unsupported,None,None
2,30,data_type,pred_only,other,None,unsupported,None,None
3,30,data_type,pred_only,time_series,None,unsupported,None,None
4,9,geospatial_info_dataset,pred_only,maps,None,unsupported,None,None
5,19,geospatial_info_dataset,pred_only,maps,None,unsupported,None,None
6,19,geospatial_info_dataset,match,sample,unsupported,unsupported,None,None
7,27,geospatial_info_dataset,pred_only,maps,None,unsupported,None,None
8,29,geospatial_info_dataset,pred_only,distribution,None,unsupported,None,None
9,29,geospatial_info_dataset,pred_only,sample,None,unsupported,None,None
